### Setup

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "main.py").exists()
            and (candidate / "utils").exists()
            and (candidate / "datasets").exists()
        ):
            return candidate

    raise RuntimeError("PyCIL project root not found.")


PROJECT_ROOT = find_project_root()

METADATA_ROOT = PROJECT_ROOT / "datasets" / "metadata"
SPLIT_ROOT = PROJECT_ROOT / "splits"

AID_SPLIT_ROOT = SPLIT_ROOT / "aid" / "seed_1993"
NWPU_SPLIT_ROOT = SPLIT_ROOT / "nwpu" / "seed_1993"

AID_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
NWPU_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Metadata     :", METADATA_ROOT)
print("Split root   :", SPLIT_ROOT)

Project root : C:\Skripsi\Project\PyCIL
Metadata     : C:\Skripsi\Project\PyCIL\datasets\metadata
Split root   : C:\Skripsi\Project\PyCIL\splits


### Seed Configuration

In [2]:
SPLIT_SEED = 1993
CLASS_ORDER_SEED = 1993

TRAIN_RATIO = 0.80
TEST_RATIO = 0.20

print("Split seed       :", SPLIT_SEED)
print("Class-order seed :", CLASS_ORDER_SEED)

Split seed       : 1993
Class-order seed : 1993


### Load Manifest

In [3]:
aid_df = pd.read_csv(
    METADATA_ROOT / "aid_raw_manifest.csv"
)

nwpu_df = pd.read_csv(
    METADATA_ROOT / "nwpu_raw_manifest.csv"
)

print("AID :", aid_df.shape)
print("NWPU:", nwpu_df.shape)

display(aid_df.head())

AID : (10000, 13)
NWPU: (31500, 13)


,dataset,relative_path,class_name,filename,extension,width,height,mode,format,file_size_bytes,is_valid,original_class_id,sha256
0,AID,Airport\airport_1.jpg,Airport,airport_1.jpg,.jpg,600,600,RGB,JPEG,437084,True,0,60b30a7f1b710400d5231258d66a2cee66dcf65b32dea4...
1,AID,Airport\airport_10.jpg,Airport,airport_10.jpg,.jpg,600,600,RGB,JPEG,445760,True,0,fc7b3c5e0beac3c58328b702f3bc79ab6a43923d0d19fd...
2,AID,Airport\airport_100.jpg,Airport,airport_100.jpg,.jpg,600,600,RGB,JPEG,328171,True,0,b20cbbcd31162b619a41c553ad7d4779d45e6e15f600d6...
3,AID,Airport\airport_101.jpg,Airport,airport_101.jpg,.jpg,600,600,RGB,JPEG,363411,True,0,2b1053fd06dc3f9463281d6f841c14b9290d5fa686978a...
4,AID,Airport\airport_102.jpg,Airport,airport_102.jpg,.jpg,600,600,RGB,JPEG,300199,True,0,d9b78b1b1c0761d934e948074baf5cd074172aeecade92...


In [4]:
required_columns = {
    "relative_path",
    "class_name",
    "original_class_id",
    "sha256"
}

assert required_columns.issubset(aid_df.columns)
assert required_columns.issubset(nwpu_df.columns)

print("Required manifest columns: OK")

Required manifest columns: OK


### Stratified 80/20 split

In [5]:
def stratified_split_by_class(
    df,
    train_ratio=0.8,
    seed=1993
):
    rng = np.random.RandomState(seed)

    parts = []

    # Sorting makes iteration deterministic
    class_ids = sorted(
        df["original_class_id"].unique()
    )

    for class_id in class_ids:
        class_df = (
            df[
                df["original_class_id"] == class_id
            ]
            .sort_values("relative_path")
            .copy()
        )

        indices = np.arange(len(class_df))
        rng.shuffle(indices)

        class_df = (
            class_df
            .iloc[indices]
            .reset_index(drop=True)
        )

        n_total = len(class_df)

        n_train = int(
            round(n_total * train_ratio)
        )

        class_df["split"] = "test"
        class_df.loc[
            :n_train - 1,
            "split"
        ] = "train"

        parts.append(class_df)

    result = pd.concat(
        parts,
        ignore_index=True
    )

    return result

### Generate initial split

In [6]:
aid_split = stratified_split_by_class(
    aid_df,
    train_ratio=TRAIN_RATIO,
    seed=SPLIT_SEED
)

nwpu_split = stratified_split_by_class(
    nwpu_df,
    train_ratio=TRAIN_RATIO,
    seed=SPLIT_SEED
)

In [7]:
print("AID:")
print(aid_split["split"].value_counts())

print()

print("NWPU:")
print(nwpu_split["split"].value_counts())

AID:
split
train    8000
test     2000
Name: count, dtype: int64

NWPU:
split
train    25200
test      6300
Name: count, dtype: int64


### Class distribution check

In [8]:
aid_split_counts = pd.crosstab(
    aid_split["class_name"],
    aid_split["split"]
)

aid_split_counts["total"] = (
    aid_split_counts["train"]
    + aid_split_counts["test"]
)

aid_split_counts["train_ratio"] = (
    aid_split_counts["train"]
    / aid_split_counts["total"]
)

display(aid_split_counts)

split,test,train,total,train_ratio
class_name,,,,
Airport,72,288,360,0.8
BareLand,62,248,310,0.8
BaseballField,44,176,220,0.8
Beach,80,320,400,0.8
Bridge,72,288,360,0.8
Center,52,208,260,0.8
Church,48,192,240,0.8
Commercial,70,280,350,0.8
DenseResidential,82,328,410,0.8


In [9]:
print(
    "AID train-ratio range:",
    aid_split_counts["train_ratio"].min(),
    "-",
    aid_split_counts["train_ratio"].max()
)

AID train-ratio range: 0.8 - 0.8


In [10]:
nwpu_split_counts = pd.crosstab(
    nwpu_split["class_name"],
    nwpu_split["split"]
)

nwpu_split_counts["total"] = (
    nwpu_split_counts["train"]
    + nwpu_split_counts["test"]
)

display(nwpu_split_counts.head(10))

split,test,train,total
class_name,,,
airplane,140,560,700
airport,140,560,700
baseball_diamond,140,560,700
basketball_court,140,560,700
beach,140,560,700
bridge,140,560,700
chaparral,140,560,700
church,140,560,700
circular_farmland,140,560,700


In [11]:
assert (
    nwpu_split_counts["train"] == 560
).all()

assert (
    nwpu_split_counts["test"] == 140
).all()

print("NWPU exact 560/140 split per class: PASSED")

NWPU exact 560/140 split per class: PASSED


### Check if NWPU duplicate is seperated from train and test

In [12]:
def find_cross_split_duplicates(df):
    split_count = (
        df.groupby("sha256")["split"]
        .nunique()
    )

    crossing_hashes = split_count[
        split_count > 1
    ].index

    return (
        df[
            df["sha256"].isin(crossing_hashes)
        ]
        .sort_values(
            ["sha256", "relative_path"]
        )
        .copy()
    )

In [13]:
nwpu_cross_split = find_cross_split_duplicates(
    nwpu_split
)

print(
    "NWPU duplicate rows crossing train/test:",
    len(nwpu_cross_split)
)

display(nwpu_cross_split)

NWPU duplicate rows crossing train/test: 6


,dataset,relative_path,class_name,filename,extension,width,height,mode,format,file_size_bytes,is_valid,original_class_id,sha256,split
776,NWPU-RESISC45,airport\airport_118.jpg,airport,airport_118.jpg,.jpg,256,256,RGB,JPEG,12491,True,1,0a4d80dd498029f027c9e8ec5cbff130d48365e7adfe56...,train
1316,NWPU-RESISC45,airport\airport_315.jpg,airport,airport_315.jpg,.jpg,256,256,RGB,JPEG,12491,True,1,0a4d80dd498029f027c9e8ec5cbff130d48365e7adfe56...,test
912,NWPU-RESISC45,airport\airport_050.jpg,airport,airport_050.jpg,.jpg,256,256,RGB,JPEG,15122,True,1,61b6a3d9edd9c1ff6c4fbb07a5bf8e1b0812a9cad6da09...,train
1262,NWPU-RESISC45,airport\airport_420.jpg,airport,airport_420.jpg,.jpg,256,256,RGB,JPEG,15122,True,1,61b6a3d9edd9c1ff6c4fbb07a5bf8e1b0812a9cad6da09...,test
1378,NWPU-RESISC45,airport\airport_243.jpg,airport,airport_243.jpg,.jpg,256,256,RGB,JPEG,14223,True,1,b0e156fd5615364a95294af398b503c3f3abd34f40050b...,test
1116,NWPU-RESISC45,airport\airport_397.jpg,airport,airport_397.jpg,.jpg,256,256,RGB,JPEG,14223,True,1,b0e156fd5615364a95294af398b503c3f3abd34f40050b...,train


### Duplicate repair

In [16]:
def repair_cross_split_duplicates(df):
    df = df.copy()

    crossing = find_cross_split_duplicates(df)

    if len(crossing) == 0:
        print("No duplicate leakage detected.")
        return df

    crossing_hashes = (
        crossing["sha256"]
        .unique()
    )

    for duplicate_hash in crossing_hashes:

        duplicate_rows = df[
            df["sha256"] == duplicate_hash
        ]

        class_ids = (
            duplicate_rows[
                "original_class_id"
            ]
            .unique()
        )

        assert len(class_ids) == 1

        class_id = class_ids[0]

        test_duplicate_indices = duplicate_rows[
            duplicate_rows["split"] == "test"
        ].index

        n_to_move = len(
            test_duplicate_indices
        )

        if n_to_move == 0:
            continue

        # Move duplicates into train
        df.loc[
            test_duplicate_indices,
            "split"
        ] = "train"

        # Select unique samples from the same class
        # to move train -> test
        duplicate_hash_counts = (
            df["sha256"]
            .value_counts()
        )

        unique_hashes = set(
            duplicate_hash_counts[
                duplicate_hash_counts == 1
            ].index
        )

        candidates = df[
            (df["original_class_id"] == class_id)
            &
            (df["split"] == "train")
            &
            (df["sha256"].isin(unique_hashes))
        ].sort_values(
            "relative_path"
        )

        assert len(candidates) >= n_to_move

        replacement_indices = (
            candidates
            .head(n_to_move)
            .index
        )

        df.loc[
            replacement_indices,
            "split"
        ] = "test"

    return df

In [17]:
nwpu_split = repair_cross_split_duplicates(
    nwpu_split
)

In [18]:
nwpu_cross_split_after = (
    find_cross_split_duplicates(
        nwpu_split
    )
)

print(
    "Cross-split duplicates after repair:",
    len(nwpu_cross_split_after)
)

assert len(
    nwpu_cross_split_after
) == 0

Cross-split duplicates after repair: 0


### Assure NWPU class count doesnt change

In [19]:
nwpu_final_counts = pd.crosstab(
    nwpu_split["class_name"],
    nwpu_split["split"]
)

assert (
    nwpu_final_counts["train"] == 560
).all()

assert (
    nwpu_final_counts["test"] == 140
).all()

assert len(nwpu_split) == 31500

print("NWPU final split validation: PASSED")

NWPU final split validation: PASSED


### Global Leakage validation

In [20]:
def validate_split(
    split_df,
    expected_total,
    expected_classes
):
    train_df = split_df[
        split_df["split"] == "train"
    ]

    test_df = split_df[
        split_df["split"] == "test"
    ]

    # Every image appears exactly once
    assert len(split_df) == expected_total

    assert (
        split_df["relative_path"]
        .nunique()
        == expected_total
    )

    # Correct class count
    assert (
        split_df["original_class_id"]
        .nunique()
        == expected_classes
    )

    # Train/test paths cannot overlap
    train_paths = set(
        train_df["relative_path"]
    )

    test_paths = set(
        test_df["relative_path"]
    )

    assert train_paths.isdisjoint(
        test_paths
    )

    # SHA hashes cannot cross train/test
    train_hashes = set(
        train_df["sha256"]
    )

    test_hashes = set(
        test_df["sha256"]
    )

    assert train_hashes.isdisjoint(
        test_hashes
    )

    print("Validation PASSED")

In [21]:
print("AID")
validate_split(
    aid_split,
    expected_total=10000,
    expected_classes=30
)

print()

print("NWPU")
validate_split(
    nwpu_split,
    expected_total=31500,
    expected_classes=45
)

AID
Validation PASSED

NWPU
Validation PASSED


### Seperate train/test manifest

In [22]:
MANIFEST_COLUMNS = [
    "relative_path",
    "class_name",
    "original_class_id",
    "sha256",
]

In [23]:
aid_train = (
    aid_split[
        aid_split["split"] == "train"
    ][MANIFEST_COLUMNS]
    .sort_values(
        ["original_class_id", "relative_path"]
    )
    .reset_index(drop=True)
)

aid_test = (
    aid_split[
        aid_split["split"] == "test"
    ][MANIFEST_COLUMNS]
    .sort_values(
        ["original_class_id", "relative_path"]
    )
    .reset_index(drop=True)
)

In [24]:
nwpu_train = (
    nwpu_split[
        nwpu_split["split"] == "train"
    ][MANIFEST_COLUMNS]
    .sort_values(
        ["original_class_id", "relative_path"]
    )
    .reset_index(drop=True)
)

nwpu_test = (
    nwpu_split[
        nwpu_split["split"] == "test"
    ][MANIFEST_COLUMNS]
    .sort_values(
        ["original_class_id", "relative_path"]
    )
    .reset_index(drop=True)
)

### Save manifest

In [25]:
aid_train.to_csv(
    AID_SPLIT_ROOT / "train.csv",
    index=False
)

aid_test.to_csv(
    AID_SPLIT_ROOT / "test.csv",
    index=False
)

nwpu_train.to_csv(
    NWPU_SPLIT_ROOT / "train.csv",
    index=False
)

nwpu_test.to_csv(
    NWPU_SPLIT_ROOT / "test.csv",
    index=False
)

print("AID train :", len(aid_train))
print("AID test  :", len(aid_test))

print()

print("NWPU train:", len(nwpu_train))
print("NWPU test :", len(nwpu_test))

AID train : 8000
AID test  : 2000

NWPU train: 25200
NWPU test : 6300


### Generate class order per dataset

In [26]:
def generate_class_order(
    num_classes,
    seed=1993
):
    rng = np.random.RandomState(seed)

    return rng.permutation(
        num_classes
    ).tolist()


aid_class_order = generate_class_order(
    30,
    CLASS_ORDER_SEED
)

nwpu_class_order = generate_class_order(
    45,
    CLASS_ORDER_SEED
)

print("AID class order:")
print(aid_class_order)

print("\nNWPU class order:")
print(nwpu_class_order)

AID class order:
[27, 21, 26, 2, 8, 13, 10, 6, 15, 20, 14, 23, 12, 28, 11, 5, 0, 22, 18, 4, 19, 24, 9, 7, 16, 3, 29, 17, 25, 1]

NWPU class order:
[21, 35, 13, 17, 25, 18, 10, 22, 41, 8, 2, 38, 27, 1, 16, 37, 0, 30, 43, 12, 6, 15, 20, 24, 19, 14, 34, 26, 36, 23, 7, 40, 28, 11, 5, 32, 4, 9, 42, 39, 31, 3, 29, 44, 33]


### Task generator function

In [27]:
def build_tasks(
    class_order,
    init_cls,
    increment
):
    tasks = []

    start = 0
    task_id = 0

    first_task = class_order[
        :init_cls
    ]

    tasks.append({
        "task_id": task_id,
        "original_class_ids": first_task
    })

    start = init_cls
    task_id += 1

    while start < len(class_order):
        task_classes = class_order[
            start:start + increment
        ]

        tasks.append({
            "task_id": task_id,
            "original_class_ids": task_classes
        })

        start += increment
        task_id += 1

    return tasks

### Make T=5 and T=10

In [28]:
aid_T5_tasks = build_tasks(
    aid_class_order,
    init_cls=14,
    increment=4
)

aid_T10_tasks = build_tasks(
    aid_class_order,
    init_cls=12,
    increment=2
)

nwpu_T5_tasks = build_tasks(
    nwpu_class_order,
    init_cls=25,
    increment=5
)

nwpu_T10_tasks = build_tasks(
    nwpu_class_order,
    init_cls=18,
    increment=3
)

In [29]:
print(
    "AID T5:",
    [len(t["original_class_ids"])
     for t in aid_T5_tasks]
)

print(
    "AID T10:",
    [len(t["original_class_ids"])
     for t in aid_T10_tasks]
)

print(
    "NWPU T5:",
    [len(t["original_class_ids"])
     for t in nwpu_T5_tasks]
)

print(
    "NWPU T10:",
    [len(t["original_class_ids"])
     for t in nwpu_T10_tasks]
)

AID T5: [14, 4, 4, 4, 4]
AID T10: [12, 2, 2, 2, 2, 2, 2, 2, 2, 2]
NWPU T5: [25, 5, 5, 5, 5]
NWPU T10: [18, 3, 3, 3, 3, 3, 3, 3, 3, 3]


### Task Mapping

In [30]:
def id_to_class_mapping(df):
    mapping = (
        df[
            [
                "original_class_id",
                "class_name"
            ]
        ]
        .drop_duplicates()
        .set_index(
            "original_class_id"
        )["class_name"]
        .to_dict()
    )

    return mapping


aid_id_to_name = id_to_class_mapping(
    aid_df
)

nwpu_id_to_name = id_to_class_mapping(
    nwpu_df
)

In [31]:
def enrich_tasks(
    tasks,
    id_to_name
):
    result = []

    for task in tasks:
        ids = task[
            "original_class_ids"
        ]

        result.append({
            "task_id":
                task["task_id"],

            "original_class_ids":
                ids,

            "class_names":
                [
                    id_to_name[i]
                    for i in ids
                ],
        })

    return result

### Save configuration JSON

In [32]:
def save_task_config(
    path,
    dataset_name,
    seed,
    class_order,
    init_cls,
    increment,
    tasks,
    id_to_name,
):
    config = {
        "dataset": dataset_name,
        "class_order_seed": seed,
        "num_classes": len(class_order),
        "init_cls": init_cls,
        "increment": increment,
        "num_tasks": len(tasks),
        "class_order": class_order,
        "class_order_names": [
            id_to_name[i]
            for i in class_order
        ],
        "tasks": enrich_tasks(
            tasks,
            id_to_name
        ),
    }

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            config,
            f,
            indent=4
        )

In [33]:
save_task_config(
    AID_SPLIT_ROOT / "T5_tasks.json",
    "AID",
    CLASS_ORDER_SEED,
    aid_class_order,
    14,
    4,
    aid_T5_tasks,
    aid_id_to_name,
)

save_task_config(
    AID_SPLIT_ROOT / "T10_tasks.json",
    "AID",
    CLASS_ORDER_SEED,
    aid_class_order,
    12,
    2,
    aid_T10_tasks,
    aid_id_to_name,
)

save_task_config(
    NWPU_SPLIT_ROOT / "T5_tasks.json",
    "NWPU-RESISC45",
    CLASS_ORDER_SEED,
    nwpu_class_order,
    25,
    5,
    nwpu_T5_tasks,
    nwpu_id_to_name,
)

save_task_config(
    NWPU_SPLIT_ROOT / "T10_tasks.json",
    "NWPU-RESISC45",
    CLASS_ORDER_SEED,
    nwpu_class_order,
    18,
    3,
    nwpu_T10_tasks,
    nwpu_id_to_name,
)

print("Task configurations saved.")

Task configurations saved.


### Validate everything

In [34]:
def validate_tasks(
    tasks,
    expected_num_tasks,
    expected_num_classes
):
    all_classes = []

    for task in tasks:
        all_classes.extend(
            task["original_class_ids"]
        )

    assert len(tasks) == expected_num_tasks

    assert len(all_classes) == expected_num_classes

    assert len(set(all_classes)) == expected_num_classes

    assert set(all_classes) == set(
        range(expected_num_classes)
    )


validate_tasks(
    aid_T5_tasks,
    5,
    30
)

validate_tasks(
    aid_T10_tasks,
    10,
    30
)

validate_tasks(
    nwpu_T5_tasks,
    5,
    45
)

validate_tasks(
    nwpu_T10_tasks,
    10,
    45
)

assert aid_T5_tasks[0]["original_class_ids"] == \
       aid_class_order[:14]

assert aid_T10_tasks[0]["original_class_ids"] == \
       aid_class_order[:12]

assert nwpu_T5_tasks[0]["original_class_ids"] == \
       nwpu_class_order[:25]

assert nwpu_T10_tasks[0]["original_class_ids"] == \
       nwpu_class_order[:18]

print("FINAL SPLIT + TASK VALIDATION: PASSED")

FINAL SPLIT + TASK VALIDATION: PASSED


### Freeze the split

In [35]:
import hashlib


def file_sha256(path):
    sha = hashlib.sha256()

    with open(path, "rb") as f:
        while chunk := f.read(1024 * 1024):
            sha.update(chunk)

    return sha.hexdigest()


split_files = [
    AID_SPLIT_ROOT / "train.csv",
    AID_SPLIT_ROOT / "test.csv",
    AID_SPLIT_ROOT / "T5_tasks.json",
    AID_SPLIT_ROOT / "T10_tasks.json",

    NWPU_SPLIT_ROOT / "train.csv",
    NWPU_SPLIT_ROOT / "test.csv",
    NWPU_SPLIT_ROOT / "T5_tasks.json",
    NWPU_SPLIT_ROOT / "T10_tasks.json",
]

checksums = {
    str(path.relative_to(PROJECT_ROOT)): file_sha256(path)
    for path in split_files
}

checksum_path = SPLIT_ROOT / "split_checksums.json"

with open(checksum_path, "w", encoding="utf-8") as f:
    json.dump(checksums, f, indent=4)

print("Split checksums saved:")
display(checksums)

Split checksums saved:


{'splits\\aid\\seed_1993\\train.csv': 'e3b577f1ff6d1ce8831bc1f8bd2a019e94c4a9a40e2c0de18ec7b6a1efa2c366',
 'splits\\aid\\seed_1993\\test.csv': 'e63539590dc727c0ed9e9e2bbf451305e7ce70d4f977034f77bd8577facf3c96',
 'splits\\aid\\seed_1993\\T5_tasks.json': '5c678d4bba06886f1c01af18c133dfb103b981ace89c05b3fc570edfe32c6699',
 'splits\\aid\\seed_1993\\T10_tasks.json': '9d93476bfe008ae13042751367e6bf78be012270001f83a2caec47b4142e6d4b',
 'splits\\nwpu\\seed_1993\\train.csv': '502ecc53b22b24bbfbf9ceaac39b344be891d30bf1aae94bdb2aa0baf36b46d6',
 'splits\\nwpu\\seed_1993\\test.csv': 'ac28fcc99552a48a703c8feceda04389724c33cfbc300b9b2cad657a05f79de6',
 'splits\\nwpu\\seed_1993\\T5_tasks.json': '0233cdd866b6d2ff8304b870aed4de9a4fc663f1a201e0084708c242b7dd8a72',
 'splits\\nwpu\\seed_1993\\T10_tasks.json': '1eb173ed14a727d9db61cf45c66426ddc235b790260059ff9cf7b785933f861f'}